In [208]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("Data")

training_files = [
    data_dir / 'CRMLSSold202509.csv',
    data_dir / 'CRMLSSold202510.csv',
    data_dir / 'CRMLSSold202511.csv',
    data_dir / 'CRMLSSold202512.csv',
    data_dir / 'CRMLSSold202601.csv',
    data_dir / 'CRMLSSold202602.csv',
    data_dir / 'CRMLSSold202603.csv'
]
dfs = [pd.read_csv(f) for f in training_files]
df = pd.concat(dfs, ignore_index=True)

df = df[
    (df['PropertyType'] == 'Residential') &
    (df['PropertySubType'] == 'SingleFamilyResidence')
    ]

cols = ['ClosePrice', 'LivingArea', 'BedroomsTotal', 
        'BathroomsTotalInteger', 'LotSizeSquareFeet']

#convert columns to numeric
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

C:\Users\d_sru\AppData\Local\Temp\ipykernel_47372\591390252.py:16: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(f) for f in training_files]


### Handle Missing Values

Check the null rates for the columns

In [209]:
important_col = ['ClosePrice', 'LivingArea', 'BedroomsTotal', 
        'BathroomsTotalInteger', 'LotSizeSquareFeet']

# Calculate missing count and percentage
null_counts = df[important_col].isnull().sum()
print('Missing values per column:')
print(null_counts)

null_percent = (df[important_col].isnull().sum() / len(df)) * 100
print('\nMissing % per column:')
print(null_percent)

Missing values per column:
ClosePrice                  0
LivingArea                 35
BedroomsTotal               0
BathroomsTotalInteger       2
LotSizeSquareFeet        1210
dtype: int64

Missing % per column:
ClosePrice               0.000000
LivingArea               0.049368
BedroomsTotal            0.000000
BathroomsTotalInteger    0.002821
LotSizeSquareFeet        1.706725
dtype: float64


- Drop ClosePrice and LivingArea rows since they are required for the model and cannot be estimated 
- Fill in LotSizeSquareFeet and BathroomsTotalInteger with the median since the columns are useful but missing for fewer than 50% of rows

In [210]:
important_col = ['ClosePrice', 'LivingArea', 'BedroomsTotal', 
        'BathroomsTotalInteger', 'LotSizeSquareFeet']

# Drop rows with missing target values
df = df.dropna(subset=['ClosePrice', 'LivingArea'])

for col in ['LotSizeSquareFeet', 'BathroomsTotalInteger']:
        if col in df.columns:
                df[col] = df[col].fillna(df[col].median())

#remove outliers
df = df[df['ClosePrice'] >= 50_000]
df = df[df['ClosePrice'] <= 10_000_000]

df = df[df['LivingArea'] >= 300]
df = df[df['LivingArea'] <= 15_000]

print('Shape after cleaning:', df.shape)

Shape after cleaning: (70538, 78)


Check other features to include to better predict ClosePrice

In [211]:
zip_median = df.groupby('PostalCode')['ClosePrice'].median().sort_values()
city_median = df.groupby('City')['ClosePrice'].median().sort_values()

print('ZIP code price range:')
print('Cheapest:', zip_median.head(3))
print('\nMost expensive:', zip_median.tail(3))

print('\nCity price range:')
print('Cheapest:', city_median.head(3))
print('\nMost expensive:', city_median.tail(3))

ZIP code price range:
Cheapest: PostalCode
92347    75000.0
93255    80000.0
93205    80000.0
Name: ClosePrice, dtype: float64

Most expensive: PostalCode
95003    7424099.0
92075    8200000.0
90020    9000000.0
Name: ClosePrice, dtype: float64

City price range:
Cheapest: City
Hinkley    75000.0
Onyx       80000.0
Bodfish    80000.0
Name: ClosePrice, dtype: float64

Most expensive: City
Bel Air         5775000.0
Hidden Hills    6650000.0
Montecito       6875000.0
Name: ClosePrice, dtype: float64


PostalCode and City both influence ClosePrice:
- ZIP code median prices range from $75K to $9M
- City median prices range from $75K to $6.9M

Since both features are important to the prediction of ClosePrice they should be including as necessary features for training.

In [212]:
#clean the PostalCode and City columns by converting them to string type and keeping the postal code to 5 digits
df['PostalCode5'] = df['PostalCode'].astype(str).str.slice(0, 5)
df['City'] = df['City'].astype(str)

# Calculate the median ClosePrice for each ZIP code and city
df['zip_median_price'] = df.groupby('PostalCode5')['ClosePrice'].transform('median')
df['city_median_price'] = df.groupby('City')['ClosePrice'].transform('median')

### Create Test Split

In [213]:
# Test set is April 2026 data
test_df = pd.read_csv('Data/CRMLSSold202604.csv', low_memory=False)
test_df = test_df[
    (test_df['PropertyType'] == 'Residential') &
    (test_df['PropertySubType'] == 'SingleFamilyResidence')
].copy()

feature_cols = ['ClosePrice','LivingArea','BedroomsTotal',
                'BathroomsTotalInteger','LotSizeSquareFeet', 'PostalCode', 'City' ]

test_df = test_df[feature_cols].copy()

cols = ['ClosePrice', 'LivingArea', 'BedroomsTotal', 
        'BathroomsTotalInteger', 'LotSizeSquareFeet']

#convert columns to numeric
for col in cols:
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

test_df = test_df.dropna(subset=['ClosePrice', 'LivingArea'])
test_df = test_df[test_df['ClosePrice'].between(50_000, 10_000_000)]
test_df = test_df[test_df['LivingArea'].between(300, 15_000)]
print('Test shape after outlier removal:', test_df.shape)

for col in ['LotSizeSquareFeet', 'BathroomsTotalInteger']:
        if col in test_df.columns:
                test_df[col] = test_df[col].fillna(test_df[col].median())

test_df['PostalCode5'] = test_df['PostalCode'].astype(str).str.slice(0, 5)
test_df['City'] = test_df['City'].astype(str)

zip_lookup = df.groupby('PostalCode5')['ClosePrice'].median()
city_lookup = df.groupby('City')['ClosePrice'].median()

global_median = df['ClosePrice'].median()

# Map training medians onto test set - never recompute from test data
test_df['zip_median_price'] = (test_df['PostalCode5'].map(zip_lookup).fillna(global_median))
test_df['city_median_price'] = (test_df['City'].map(city_lookup).fillna(global_median))

final_feature_cols = ['ClosePrice','LivingArea','BedroomsTotal','BathroomsTotalInteger',
                      'LotSizeSquareFeet', 'zip_median_price', 'city_median_price' ]

test_df = test_df[final_feature_cols].copy()
df = df[final_feature_cols].copy()

print('Test set shape:', test_df.shape)
print('Training set shape:', df.shape)

Test shape after outlier removal: (11969, 7)
Test set shape: (11969, 7)
Training set shape: (70538, 7)


### Save Cleaned CSVs

In [214]:
df.to_csv('cleaned_training.csv', index=False)
test_df.to_csv('cleaned_test.csv', index=False)

print('Saved cleaned_training.csv')
print('Saved cleaned_test.csv')
print('\nFinal training shape:', df.shape)
print('Final test shape:', test_df.shape)

Saved cleaned_training.csv
Saved cleaned_test.csv

Final training shape: (70538, 7)
Final test shape: (11969, 7)


In [215]:
feature_cols = ['LivingArea','BedroomsTotal','BathroomsTotalInteger',
                      'LotSizeSquareFeet', 'zip_median_price', 'city_median_price' ]

X_train = df[feature_cols].copy()
y_train = df['ClosePrice'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['ClosePrice'].copy()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (70538, 6) | y_train shape: (70538,)
X_test shape:  (11969, 6)  | y_test shape:  (11969,)
